In [0]:
# Setup_infrastructure.py
# 1. Run this cell first and enter in the widged above the your azure blob storage path abfss://<container_name>@<storage_account_name>.dfs.core.windows.net/

import pyspark.dbutils 

# 1. Define the widget (This creates the UI box)
dbutils.widgets.text("storage_path", "", "Azure ABFSS path here")

In [0]:
# Run this cell to create prod/dev/test catalogs and medallion schemas

storage_root = dbutils.widgets.get("storage_path")

if not storage_root:
    raise ValueError("Please provide a storage_path widget!")

envs = {
    "prod": "prod_catalog",
    "dev": "dev_catalog",
    "test": "test_catalog"
}

for env, catalog in envs.items():

    catalog_path = f"{storage_root}/managed/data/{env}/"

    # Create catalog
    spark.sql(f"""
        CREATE CATALOG IF NOT EXISTS {catalog}
        MANAGED LOCATION '{catalog_path}'
    """)

    # Create schemas
    for layer in ["bronze", "silver", "gold"]:
        spark.sql(f"""
            CREATE SCHEMA IF NOT EXISTS {catalog}.{layer}
        """)

print(f"Infrastructure created successfully using root: {storage_root}")